# SOFA trajectory forecasting — data preparation, model development & evaluation

**TRIPOD+AI items 1, 2, 3a–3b, 4.** This is the single, canonical notebook for the project:
it develops a model that forecasts a patient's rolling SOFA (organ-failure) score every 6 h
across ICU hours 24–48, from the PhysioNet/CinC Challenge 2012 dataset, and reports how well
that model performs on data it never trained on. It supersedes the earlier
`phase1_data.ipynb` (folded in below) and is the companion artefact to
**`SOFA_forecast_project_plan.docx`**, which holds the full problem statement (§1), decision
log (§3) and the TRIPOD+AI compliance table (§7) this notebook's headers cross-reference.

**Item 3a–3b — clinical context & target population.** Organ dysfunction in ICU patients
changes hour to hour; SOFA is normally scored once every 24 h, retrospectively. This model
gives a clinician, at any 6-hour checkpoint from hour 24 onward, an estimate of where the
patient's SOFA will be a further 6 h out — turning a retrospective score into a short-horizon
early-warning signal. Target population: adult ICU stays ≥ 48 h (PhysioNet/CinC 2012 set A,
`n = 12,000`).

**Item 4 — study objective.** Model **development** with **internal** validation (train/val
split) and a single held-out **internal test** evaluation. No external (different-hospital or
different-era) validation is available or attempted — logged as a limitation (§14 below).

**Status of this notebook.** Executed end to end, including training all five model
families (CatBoost/LightGBM installed on the training device) — every number below,
including Results (§11), Interpretation (§12) and Discussion (§13), is real. Two real bugs
were found and fixed only once this was actually run (D23, D24 — see the decision log);
neither is visible from reading the code alone, which is exactly why this run matters.


In [2]:
import os, sys, time, pathlib
sys.path.insert(0, os.path.abspath(".."))          # project root on path

import numpy as np, pandas as pd
from scipy import stats
import joblib

from modeling import config
from modeling.data_prep import parsing, splits, targets, features, qc
from modeling import models, evaluate

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)
print("horizons:", config.HORIZONS_H)
for T in config.HORIZONS_H:
    print(f"  T={T:>2}h  features t<{config.feature_cutoff_min(T)}min  "
          f"target window {tuple(x//60 for x in config.target_window_min(T))}h  "
          f"now window {tuple(x//60 for x in config.now_window_min(T))}h")


horizons: (24, 30, 36, 42)
  T=24h  features t<1440min  target window (6, 30)h  now window (0, 24)h
  T=30h  features t<1800min  target window (12, 36)h  now window (6, 30)h
  T=36h  features t<2160min  target window (18, 42)h  now window (12, 36)h
  T=42h  features t<2520min  target window (24, 48)h  now window (18, 42)h


## 1&nbsp;&nbsp;Data source & participants&nbsp;&nbsp;*(TRIPOD+AI 5a–5b, 6a–6c, 7)*

**5a — source of data.** PhysioNet/Computing in Cardiology Challenge 2012, set A: 12,000 ICU
stays, publicly available, de-identified. This project uses no other data source.
**5b — accrual dates.** Not available in the de-identified release — logged as a limitation
(no admission calendar dates, so temporal drift/seasonality cannot be assessed).
**6a — setting.** Four ICU types (coronary care, cardiac-surgery recovery, medical, surgical).
**6b — eligibility.** The dataset's own inclusion rule: adult ICU stay, length of stay ≥ 48 h
(shorter stays already excluded by PhysioNet before release). No further exclusions are
applied here — all 12,000 records are used.
**6c — treatments received.** No medication data of any kind exists in this dataset —
notably **no vasopressor/inotrope name or dose**, which is why the SOFA cardiovascular axis
is capped at 0/1 throughout this project (quantified in `SOFA_modelling_approaches.docx`).
**7 — pre-processing/QC.** `data_prep/parsing.py` applies the cleaning rules below; the audit
must reproduce `sofa_calculation.ipynb`'s section-2 counts exactly.


In [3]:
t0 = time.time()
if config.RECORDS_CLEAN.exists():
    records = joblib.load(config.RECORDS_CLEAN)
    print(f"loaded {len(records)} cleaned records from cache in {time.time()-t0:.1f}s")
else:
    records = parsing.parse_all()
    joblib.dump(records, config.RECORDS_CLEAN, compress=3)
    print(f"parsed + cached {len(records)} records in {time.time()-t0:.1f}s")

audit = qc.audit_raw()
bad = {k: (audit['counts'][k], v) for k, v in qc.DOC_REF.items() if audit['counts'][k] != v}
print("section-7 audit mismatches vs doc:", bad or "NONE -- all match")
pd.DataFrame({"computed": audit['counts'], "doc": qc.DOC_REF}).assign(
    ok=lambda d: d.computed == d.doc)


loaded 12000 cleaned records from cache in 11.7s
section-7 audit mismatches vs doc: NONE -- all match


,computed,doc,ok
empty_param_rows,2403,2403,True
MAP_eq_0,86,86,True
PaO2_eq_0,3,3,True
SaO2_eq_0,2,2,True
Weight_neg1,1001,1001,True
Weight_le20,353,353,True
Weight_gt300,1,1,True
Height_lt100,17,17,True
Height_gt250,11,11,True
MechVent_rows,91644,91644,True


In [4]:
age = pd.Series({rid: d.get("Age", np.nan) for rid, (_s, d) in records.items()})
gender = pd.Series({rid: d.get("Gender", np.nan) for rid, (_s, d) in records.items()})
icutype = pd.Series({rid: d.get("ICUType", np.nan) for rid, (_s, d) in records.items()})
print(f"Age:    median {age.median():.0f}  IQR {age.quantile(.25):.0f}-{age.quantile(.75):.0f}")
print(f"Gender: male {100*(gender==1).mean():.1f}%  female {100*(gender==0).mean():.1f}%")
print("ICUType (1 CCU / 2 CardSurg / 3 MICU / 4 SICU):",
      icutype.value_counts().sort_index().to_dict())

lt = audit['last_t_hours']
print(f"\nlast observation (h): min {lt.min():.1f}  p5 {np.percentile(lt,5):.1f}  "
      f"median {np.percentile(lt,50):.1f}  max {lt.max():.1f}")
print("records ending before 42h:", int((lt < 42).sum()), " before 36h:", int((lt < 36).sum()))


Age:    median 67  IQR 53-78
Gender: male 56.1%  female 43.8%
ICUType (1 CCU / 2 CardSurg / 3 MICU / 4 SICU): {1.0: 1769, 2.0: 2530, 3.0: 4293, 4.0: 3408}

last observation (h): min 0.0  p5 46.3  median 47.5  max 48.0
records ending before 42h: 171  before 36h: 83


## 2&nbsp;&nbsp;Outcome definition&nbsp;&nbsp;*(TRIPOD+AI 8a–8c)*

**8a — outcome & time horizon.** The outcome is the **rolling SOFA score**: for a decision
time `T ∈ {24,30,36,42} h`, the outcome is the six-axis SOFA total (and per-axis sub-scores)
over the trailing-24 h window `[T−18, T+6)h` — i.e. the score **6 h ahead** of `T`. A
secondary binary outcome, `deteriorate_24h`, is `1` when that score rises ≥ 2 points versus
the score on the last fully-observed 24 h (`SOFA_now`). Both outcomes are computed
**algorithmically** by the project's rule engine (`data_prep/scoring.py`, unchanged from
`sofa_calculation.ipynb`) — never `outcomes.csv`, which is the outcome the challenge
organisers derived with data (vasopressor doses) this project does not have, and is used only
as an external reference in evaluation, never as a training signal.
**8b–8c — assessor blinding.** Not applicable: the outcome is computed by a fixed,
version-controlled scoring function, not by a human rater.


In [5]:
t0 = time.time()
tg = targets.build_targets(records)
tg.to_csv(config.TARGETS_CSV, index=False)
print(f"targets {tg.shape} in {time.time()-t0:.1f}s   rows/record: "
      f"{tg.groupby('RecordID').size().value_counts().to_dict()}")
tg.groupby("origin_h")[["sofa_total","sofa_now","sofa_delta","deteriorate_24h",
                        "cardio_instability","target_window_empty"]].mean().round(3)


targets (48000, 28) in 3.9s   rows/record: {4: 12000}


,sofa_total,sofa_now,sofa_delta,deteriorate_24h,cardio_instability,target_window_empty
origin_h,,,,,,
24,5.275,5.747,-0.483,0.046,0.684,0.002
30,4.982,5.264,-0.294,0.038,0.653,0.002
36,4.810,4.970,-0.172,0.042,0.627,0.002
42,4.631,4.798,-0.179,0.039,0.603,0.003


In [6]:
# sanity: the T=42 target window IS Day 2 [24,48)h -> must equal notebook SOFA_D2
sc = pd.read_csv(config.SOFA_SCORES_CSV)[["RecordID","SOFA_D2"]]
chk = tg[tg.origin_h == 42].merge(sc, on="RecordID")
d = chk.sofa_total.fillna(0) - chk.SOFA_D2
r = stats.pearsonr(chk.sofa_total.fillna(0), chk.SOFA_D2)[0]
print(f"T=42 sofa_total vs sofa_scores.SOFA_D2:  r = {r:.4f}   mean|diff| = {d.abs().mean():.4f}")
assert r > 0.98

oc = pd.read_csv(config.OUTCOMES_CSV)[["RecordID","SOFA"]]
ref = tg[tg.origin_h == 24].merge(oc, on="RecordID"); ref = ref[ref.SOFA != -1]
b = (ref.sofa_now - ref.SOFA).mean()  # sofa_now at T=24 == SOFA_D1
rr = stats.pearsonr(ref.sofa_now, ref.SOFA)[0]
print(f"[0,24)h rule SOFA (sofa_now @ T=24) vs outcomes.SOFA (reference only, never a feature): "
      f"r={rr:.3f} bias={b:+.3f}  (expect r 0.933, bias -0.862)")


T=42 sofa_total vs sofa_scores.SOFA_D2:  r = 1.0000   mean|diff| = 0.0000
[0,24)h rule SOFA (sofa_now @ T=24) vs outcomes.SOFA (reference only, never a feature): r=0.933 bias=-0.862  (expect r 0.933, bias -0.862)


## 3&nbsp;&nbsp;Predictors&nbsp;&nbsp;*(TRIPOD+AI 9a–9c)*

**9a — choice of predictors.** All time-varying vitals/labs in `icu_records` (36 variables:
GCS, HR, blood pressures, temperature, urine, chemistry panel, blood gases, MechVent,
bilirubin, etc.), summarised over `[0, T)` (last/first/min/max/mean/std/count/hours-since-last/
hours-of-coverage/12 h-slope), plus engineered hemodynamic/neuro/renal/respiratory signals,
the rule-based SOFA sub-scores computed from our own data on `[0,24)h` and `[T−24,T)h`, and
admission descriptors. No predictor pre-selection beyond excluding `outcomes.csv` entirely
(D6). **9b — measurement.** All predictors are derived only from measurements timestamped
`< T` — never the target window `[T,T+6)` (verified below). **9c** — not applicable; no
subjectively-assessed predictor.


In [7]:
t0 = time.time()
if config.FEATURES_CSV.exists():
    ft = pd.read_csv(config.FEATURES_CSV)
    print(f"loaded features {ft.shape} from cache in {time.time()-t0:.1f}s")
else:
    ft = features.build_features(records)
    ft.to_csv(config.FEATURES_CSV, index=False)
    print(f"built + cached features {ft.shape} in {time.time()-t0:.0f}s")
fam = pd.Series([c.split('_')[0] for c in ft.columns if c not in ('RecordID','origin_h')]).value_counts()
print(f"{ft.shape[1]-2} feature columns; families:", fam.to_dict())


loaded features (48000, 409) from cache in 2.4s
407 feature columns; families: {'v': 360, 'e': 14, 'sofa0': 14, 'sofa': 14, 'd': 5}


## 4&nbsp;&nbsp;Sample size&nbsp;&nbsp;*(TRIPOD+AI 10)*

**Item 10 — how the study size was arrived at.** This is a **secondary analysis of a fixed
public dataset**, not a prospectively powered study: the sample size (12,000 stays × 4
decision times = 48,000 (record × horizon) rows) is simply everything the dataset provides.
No formal power calculation was performed; the number of usable outcome events per axis per
horizon is reported below (Section 8) so readers can judge estimate precision themselves —
particularly for the rare `deteriorate_24h` outcome (~4 % positive).


## 5&nbsp;&nbsp;Missing data&nbsp;&nbsp;*(TRIPOD+AI 11)*

Two independent missingness mechanisms are handled differently, by design:

* **Predictor missingness** (a vital/lab never measured in `[0,T)`, e.g. no arterial blood gas)
  is left as `NaN` and passed straight to `HistGradientBoostingRegressor`/`Classifier`, which
  route missing values natively during tree splitting — no mean/median imputation, so a
  "never measured" case is not silently treated as "measured normal". Two exceptions, applied
  during feature/target construction (not here): bilirubin uses last-observation-carried-
  forward across windows (clinically not re-tested daily), and admission weight is imputed
  from the cohort's sex-specific median only for the urine-rate *feature* (flagged with
  `e_weight_imputed`).
* **Outcome missingness** (target window has no measurement for that axis) is **excluded**
  from that axis's training/evaluation rows (D17) — never zero-filled — so the model is never
  taught that "unmeasured" means "score 0". (`sofa_total`/`sofa_delta`/`deteriorate_24h`
  reported in Section 2 above *do* zero-fill missing axes, matching `sofa_calculation.ipynb`'s
  lenient convention — that is a reporting convention for the aggregate score, not a
  training target.)


In [8]:
nan_frac = ft.drop(columns=["RecordID","origin_h"]).isna().mean().sort_values()
print(f"fully-populated feature columns: {(nan_frac==0).sum()} / {len(nan_frac)}   "
      f"median NaN fraction: {nan_frac.median():.2f}")
print("10 most-missing features:")
nan_frac.tail(10)


fully-populated feature columns: 60 / 407   median NaN fraction: 0.18
10 most-missing features:


v_TroponinI_min           0.955792
v_TroponinI_first         0.955792
v_Bilirubin_slope12h      0.984458
v_AST_slope12h            0.985000
v_ALT_slope12h            0.985042
v_ALP_slope12h            0.985625
v_TroponinT_slope12h      0.985625
v_Albumin_slope12h        0.993417
v_TroponinI_slope12h      0.997042
v_Cholesterol_slope12h    0.999958
dtype: float64

## 6&nbsp;&nbsp;Data partitioning&nbsp;&nbsp;*(TRIPOD+AI 12a)*

Record-level (a stay's four horizon-rows never straddle a split boundary), stratified on a
coarse SOFA-severity band × in-hospital death (`SOFA == -1` records form their own band),
70 / 15 / 15 train / validation / test, `random_state = 42`, **frozen**. No patient linkage
exists in this dataset, so record-level is the finest grouping available (item 16: no
separate development/evaluation *sources* exist — both come from one dataset via this
random split).


In [9]:
sp = splits.make_splits()
sp.to_csv(config.SPLITS_CSV, index=False)
print(sp.split.value_counts().to_dict())
splits.split_summary(sp)


{'train': 8400, 'val': 1800, 'test': 1800}


,n,pct,sofa_mean,sofa_p25,sofa_p75,death_rate,sofa_missing,icu1,icu2,icu3,icu4
train,8400.0,70.0,6.66,3.0,9.0,0.142,282.0,0.146,0.211,0.358,0.285
val,1800.0,15.0,6.66,3.0,9.0,0.143,61.0,0.155,0.210,0.353,0.282
test,1800.0,15.0,6.67,3.0,9.0,0.142,60.0,0.148,0.210,0.362,0.280


## 7&nbsp;&nbsp;Internal-consistency & leakage checks

Not itself a TRIPOD+AI item, but the evidence for items 7/9b/12a being satisfied in practice.


In [10]:
J = ft.merge(tg, on=["RecordID","origin_h"]).merge(sp[["RecordID","split"]], on="RecordID")
print("joined:", J.shape)
print(J.groupby(["split","origin_h"]).size().unstack(), "\n")

ok = True
for v in ["v_HR_count","v_GCS_count","v_Urine_count","v_Creatinine_count"]:
    piv = ft.pivot_table(index="RecordID", columns="origin_h", values=v)
    mono = ((piv[24]<=piv[30]) & (piv[30]<=piv[36]) & (piv[36]<=piv[42])).mean()
    ok &= mono == 1.0
    print(f"  {v:20} monotone non-decreasing in T: {mono:.4f}")
assert ok

s024 = ft.pivot_table(index="RecordID", columns="origin_h", values="sofa0_24_total")
assert (s024.nunique(axis=1) == 1).all()
print("  sofa0_24_total identical across all 4 horizons: True")

leak = [c for c in ft.columns if c.lower() in
        ("sofa","saps-i","survival","length_of_stay","in-hospital_death")]
assert not leak
print("  outcomes.csv columns in features:", leak or "none")
print("  non-empty target rows all have >=1 organ scored:",
      not ((J.target_window_empty == 0) & (J.n_present_target == 0)).any())


joined: (48000, 436)
origin_h    24    30    36    42
split                           
test      1800  1800  1800  1800
train     8400  8400  8400  8400
val       1800  1800  1800  1800 

  v_HR_count           monotone non-decreasing in T: 1.0000
  v_GCS_count          monotone non-decreasing in T: 1.0000
  v_Urine_count        monotone non-decreasing in T: 1.0000
  v_Creatinine_count   monotone non-decreasing in T: 1.0000
  sofa0_24_total identical across all 4 horizons: True
  outcomes.csv columns in features: none
  non-empty target rows all have >=1 organ scored: True


## 8&nbsp;&nbsp;Exploratory analysis & baseline models&nbsp;&nbsp;*(TRIPOD+AI 12e, partial)*

Before fitting anything, establish the bar for **both** prediction tasks — a deliberate,
professional-practice step, not an afterthought (D18).

**Regression (total & per-axis SOFA).** **Persistence**: assume no change, i.e.
`SOFA_hat(T+6) = SOFA_now`. **Last-value-per-axis**: the per-axis analogue. Any trained model
that cannot beat persistence on MAE is not adding forecast value — it would just be relearning
"the future looks like the present".

**Classification (`deteriorate_24h`).** No model is fit here either — instead, the
**no-skill reference values** any classifier must be read against: AUROC of a
non-informative classifier is *always* 0.5, but **AUPRC of a non-informative classifier
equals the outcome's prevalence, not 0.5** (Saito & Rehmsmeier, *PLOS ONE* 2015) — with a
~4 % positive rate a classifier can post an AUPRC around 0.15–0.20 and *sound* good while
still being weak. `evaluate.no_skill_reference()` computes these three numbers from the
prevalence alone.


In [11]:
base = J.dropna(subset=["sofa_total"]).copy()
print("rows usable for the TOTAL-SOFA outcome, per split x horizon:")
print(base.groupby(["split","origin_h"]).size().unstack(), "\n")
print("deteriorate_24h positive-rate per split x horizon:")
print(base.groupby(["split","origin_h"]).deteriorate_24h.mean().unstack().round(3))


rows usable for the TOTAL-SOFA outcome, per split x horizon:
origin_h    24    30    36    42
split                           
test      1797  1797  1796  1797
train     8383  8381  8380  8377
val       1795  1794  1795  1795 

deteriorate_24h positive-rate per split x horizon:
origin_h     24     30     36     42
split                               
test      0.057  0.036  0.046  0.042
train     0.045  0.039  0.041  0.038
val       0.038  0.033  0.046  0.037


In [12]:
persist = evaluate.regression_report(
    base[base.split=="train"], y_col="sofa_total", pred_cols={"persistence": "sofa_now"})
print("TOTAL-SOFA persistence baseline (train split):")
persist


TOTAL-SOFA persistence baseline (train split):


,origin_h,series,n,mae,mae_lo,mae_hi,rmse,rmse_lo,rmse_hi
0,24,persistence,8383,0.933914,0.903853,0.964094,1.662901,1.616964,1.708762
1,30,persistence,8381,0.692519,0.668649,0.717817,1.274983,1.234954,1.317147
2,36,persistence,8380,0.633890,0.612885,0.655728,1.208166,1.171200,1.244753
3,42,persistence,8377,0.599379,0.577408,0.622072,1.232203,1.189359,1.277813


In [13]:
rows = []
for axis in models.AXES:
    d = base[base.split=="train"].dropna(subset=[f"{axis}_score", f"sofa_now_{axis}"])
    r = evaluate.bootstrap_ci(d[f"{axis}_score"], d[f"sofa_now_{axis}"], evaluate._mae, n_boot=500)
    rows.append({"axis": axis, "n": r["n"], "persistence_mae": r["point"],
                "mae_lo": r["ci_lo"], "mae_hi": r["ci_hi"]})
print("per-axis persistence baseline (train split):")
pd.DataFrame(rows)


per-axis persistence baseline (train split):


,axis,n,persistence_mae,mae_lo,mae_hi
0,resp,15248,0.232096,0.224485,0.240131
1,coag,31180,0.078384,0.075369,0.081751
2,liver,13508,0.033758,0.030610,0.036793
3,cardio,33007,0.082104,0.079283,0.084998
4,neuro,32986,0.299551,0.291971,0.306995
5,renal,33095,0.081916,0.078333,0.085817


In [14]:
rows = []
for T in config.HORIZONS_H:
    d = base[(base.split=="train") & (base.origin_h==T)].dropna(subset=["deteriorate_24h"])
    ns = evaluate.no_skill_reference(d.deteriorate_24h)
    rows.append({"origin_h": T, "n": len(d), **ns})
print("deterioration classifier -- no-skill reference (train split; no model fit):")
pd.DataFrame(rows).round(4)


deterioration classifier -- no-skill reference (train split; no model fit):


,origin_h,n,prevalence,auroc,auprc,brier
0,24,8383,0.0454,0.5,0.0454,0.0434
1,30,8381,0.0390,0.5,0.0390,0.0375
2,36,8380,0.0406,0.5,0.0406,0.0389
3,42,8377,0.0383,0.5,0.0383,0.0369


## 9&nbsp;&nbsp;Model development methods&nbsp;&nbsp;*(TRIPOD+AI 12b–12g, 13, 14, 15)*

**Five model families are compared head-to-head**, not because we plan to ship all five, but
because "does this extra complexity / different framing actually help" is itself a result
worth reporting honestly (professional practice; user-directed 2026-09-16). In priority order:

0. **`models.ordinal` (D19).** Each axis score is an **ordinal** variable in {0,1,2,3,4} —
   "2 is worse than 1" but not necessarily by the same margin as "3 is worse than 2." Rather
   than regressing the delta and clipping (below), this reframes each axis as 4 cumulative
   binary questions (Frank & Hall, 2001): *"is the score already above 0? above 1? above 2?
   above 3?"* — one `HistGradientBoostingClassifier` per question, predicting the **absolute**
   target-window score directly (no delta trick needed). Reconstruction:
   `E[score] = P(>0) + P(>1) + P(>2) + P(>3)`, which lands in `[0,4]` automatically.
1. **`models.linear` (D20).** Ridge regression / class-weighted logistic regression, same
   delta target and feature set as the primary family — isolates "does non-linearity help" as
   a single variable. Needs a median-impute + standardize step HistGB doesn't (logged
   difference: any gap partly reflects native-missingness-handling too, not purely linear vs
   non-linear).
2. **`models.histgb` — THE PRIMARY FAMILY (D12/D13/D14/D17).** Six pooled
   `HistGradientBoostingRegressor`s (one per organ axis, **pooled across all four horizons** —
   D12: 4× more rows per model than 24 separate axis×horizon models, and `origin_h` already
   lets one model express horizon-specific behaviour) predict the axis **delta** (D17: trained
   only on rows where that axis was actually measured, never the lenient zero-fill).
   `HistGradientBoosting` was chosen for native missing-value handling with no extra
   dependency (D8).
3. **`models.catboost_model` (D21, optional dependency).** Same delta target; native
   categorical handling for `ICUType`/`Gender`/`origin_h`; "ordered boosting" specifically
   targets the overfitting-from-target-leakage risk that plain boosting faces on the smaller
   axes (e.g. liver, ~13,500 training rows).
4. **`models.lightgbm_model` (D22, optional dependency).** Same delta target; leaf-wise growth
   as a second GBDT opinion against HistGB/CatBoost's more level-wise growth.

**12b — how predictors are handled.** Raw + engineered numeric features and `origin_h` (a
small integer feature so one model can learn horizon-dependent effects) go directly into the
tree-based families; the linear family additionally imputes (median) and standardizes.

**12c — model type, rationale, build steps, hyperparameter tuning.** Every family uses the
**same protocol**: a small fixed grid (4 points for the tree families, 3 for linear), each
point fit on `train`, scored on the **external** validation split (D13) — never a library's
internal early-stopping/validation_fraction, which would silently carve validation rows out
of `train`. Grids for every installed family are printed below.

**12d — heterogeneity of performance.** Every performance number carries a bootstrap
confidence interval (`evaluate.bootstrap_ci`) and a subgroup breakdown (Section 12).

**12e — performance measures.** MAE/RMSE per horizon/axis (regression) and AUROC/AUPRC/Brier
+ calibration (deterioration), for every trained family side by side, plus their respective
no-skill / persistence reference (D18, Section 8).

**12f — model updating.** Not applicable — first model version, no updating cycle yet.

**12g — how predictions are calculated.** Delta families (`histgb`/`linear`/`catboost`/
`lightgbm`): absolute sub-score = `sofa_now_<axis>` (a feature) + predicted delta, clipped to
`[0,4]`. Ordinal family: absolute sub-score = sum of the four cumulative probabilities
(no clipping needed). Total = sum of the six axes, for every family.

**13 — class imbalance.** Every deterioration classifier uses class-weighting (`balanced`)
because positives are ~4% of rows; model selection uses validation **AUPRC**, not AUROC.

**14 — fairness.** Partial: Section 12 reports performance stratified by ICU type and by
hemodynamic-instability status (a pressor-likelihood proxy); no formal fairness-metric
adjustment by age or sex is applied (Discussion: flagged as future work — this dataset's size,
once stratified by both age band and sex, would make such estimates unstable).

**15 — model output.** Six ordinal sub-scores (0–4 each) summed to a total SOFA (0–24) 6 h
ahead of the decision time, the signed delta versus the current SOFA, and a probability of
≥ 2-point deterioration within 24 h — reported once per model family.


In [15]:
print("installed model families:", models.AVAILABLE_FAMILIES)

print("\n--- ordinal (D19) -- 4 configs per threshold, selected by validation AUROC ---")
print(pd.DataFrame(models.ordinal.GRID))

print("\n--- linear (D20) -- Ridge grid, selected by validation MAE ---")
print(pd.DataFrame(models.linear.RIDGE_GRID))
print("linear (D20) -- Logistic grid, selected by validation AUPRC:")
print(pd.DataFrame(models.linear.LOGISTIC_GRID))

print("\n--- histgb (primary, D13) -- regressor grid, selected by validation MAE ---")
display_grid_r = pd.DataFrame(models.REGRESSOR_GRID)
print(display_grid_r)
print("histgb (primary, D14) -- classifier grid, selected by validation AUPRC:")
print(pd.DataFrame(models.CLASSIFIER_GRID))

print("\n--- catboost (D21) -- regressor grid ---",
      "(installed)" if models.catboost_model.CATBOOST_AVAILABLE else "(NOT installed here)")
print(pd.DataFrame(models.catboost_model.REGRESSOR_GRID))

print("\n--- lightgbm (D22) -- regressor grid ---",
      "(installed)" if models.lightgbm_model.LIGHTGBM_AVAILABLE else "(NOT installed here)")
print(pd.DataFrame(models.lightgbm_model.REGRESSOR_GRID))

display_grid_r


installed model families: {'ordinal': True, 'linear': True, 'histgb': True, 'catboost_model': True, 'lightgbm_model': True}

--- ordinal (D19) -- 4 configs per threshold, selected by validation AUROC ---
   learning_rate  max_leaf_nodes  max_iter  l2_regularization  random_state
0           0.05              31       300                0.0            42
1           0.05              63       300                0.0            42
2           0.10              31       300                0.0            42
3           0.10              63       300                0.0            42

--- linear (D20) -- Ridge grid, selected by validation MAE ---
   alpha
0    0.1
1    1.0
2   10.0
linear (D20) -- Logistic grid, selected by validation AUPRC:
      C class_weight  max_iter
0   0.1     balanced      2000
1   1.0     balanced      2000
2  10.0     balanced      2000

--- histgb (primary, D13) -- regressor grid, selected by validation MAE ---
   learning_rate  max_leaf_nodes  max_iter  l2_regular

,learning_rate,max_leaf_nodes,max_iter,l2_regularization,random_state
0,0.05,31,300,0.0,42
1,0.05,63,300,0.0,42
2,0.10,31,300,0.0,42
3,0.10,63,300,0.0,42


## 10&nbsp;&nbsp;Model training&nbsp;&nbsp;*(TRIPOD+AI 12c)*

**Every cell in this section is intentionally not run in this environment.** Each fits one
model family on `train`, selecting hyperparameters by `val` performance, exactly as described
in Section 9. Run this notebook top-to-bottom **on the training device** to populate this and
everything downstream. CatBoost/LightGBM cells check availability and skip with a clear
message (rather than crashing) if the package isn't installed there either.


### 10.0&nbsp;&nbsp;Ordinal (D19) — priority 1

In [16]:
ordinal_models, ordinal_params, ordinal_grids = models.ordinal.train_axis_models_ordinal(ft, tg, sp)
for axis in models.AXES:
    print(f"{axis:7} chosen thresholds -> params:", ordinal_params[axis])


resp    chosen thresholds -> params: {0: {'learning_rate': 0.05, 'max_leaf_nodes': 63, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}, 1: {'learning_rate': 0.1, 'max_leaf_nodes': 31, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}, 2: {'learning_rate': 0.05, 'max_leaf_nodes': 63, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}, 3: {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}}
coag    chosen thresholds -> params: {0: {'learning_rate': 0.05, 'max_leaf_nodes': 63, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}, 1: {'learning_rate': 0.1, 'max_leaf_nodes': 31, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}, 2: {'learning_rate': 0.1, 'max_leaf_nodes': 31, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}, 3: {'learning_rate': 0.05, 'max_leaf_nodes': 63, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}}
liver   chosen threshol

### 10.1&nbsp;&nbsp;Linear baseline (D20) — priority 2

In [17]:
linear_models, linear_params, linear_grids = models.linear.train_axis_models_linear(ft, tg, sp)
for axis in models.AXES:
    print(f"{axis:7} chosen params: {linear_params[axis]}")
    display(linear_grids[axis])

det_model_linear, det_params_linear, det_grid_linear = models.linear.train_deterioration_model_linear(ft, tg, sp)
print("deterioration (linear) chosen params:", det_params_linear)
display(det_grid_linear)


resp    chosen params: {'alpha': 10.0}


,alpha,val_mae
0,0.1,0.343275
1,1.0,0.342602
2,10.0,0.340772


coag    chosen params: {'alpha': 10.0}


,alpha,val_mae
0,0.1,0.142964
1,1.0,0.142834
2,10.0,0.142214


liver   chosen params: {'alpha': 10.0}


...

,alpha,val_mae
0,0.1,0.378395
1,1.0,0.378198
2,10.0,0.377851


renal   chosen params: {'alpha': 10.0}


,alpha,val_mae
0,0.1,0.148926
1,1.0,0.148756
2,10.0,0.148168


deterioration (linear) chosen params: {'C': 0.1, 'class_weight': 'balanced', 'max_iter': 2000}


,C,class_weight,max_iter,val_auprc
0,0.1,balanced,2000,0.193183
1,1.0,balanced,2000,0.186446
2,10.0,balanced,2000,0.181245


### 10.2&nbsp;&nbsp;HistGB (primary, D12–D14/D17)

In [18]:
axis_models, axis_params, axis_grids = models.train_axis_models(ft, tg, sp)
for axis in models.AXES:
    print(f"{axis:7} chosen params: {axis_params[axis]}")
    display(axis_grids[axis])

det_model, det_params, det_grid = models.train_deterioration_model(ft, tg, sp)
print("deterioration (histgb) chosen params:", det_params)
display(det_grid)


resp    chosen params: {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}


,learning_rate,max_leaf_nodes,max_iter,l2_regularization,random_state,val_mae
0,0.05,31,300,0.0,42,0.265386
1,0.05,63,300,0.0,42,0.271448
2,0.10,31,300,0.0,42,0.270485
3,0.10,63,300,0.0,42,0.274280


coag    chosen params: {'learning_rate': 0.05, 'max_leaf_nodes': 63, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}


,learning_rate,max_leaf_nodes,max_iter,l2_regularization,random_state,val_mae
0,0.05,31,300,0.0,42,0.088018
1,0.05,63,300,0.0,42,0.087040
2,0.10,31,300,0.0,42,0.089294
3,0.10,63,300,0.0,42,0.088731


liver   chosen params: {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}


...

,learning_rate,max_leaf_nodes,max_iter,l2_regularization,random_state,val_mae
0,0.05,31,300,0.0,42,0.235911
1,0.05,63,300,0.0,42,0.231556
2,0.10,31,300,0.0,42,0.240444
3,0.10,63,300,0.0,42,0.235436


renal   chosen params: {'learning_rate': 0.05, 'max_leaf_nodes': 63, 'max_iter': 300, 'l2_regularization': 0.0, 'random_state': 42}


,learning_rate,max_leaf_nodes,max_iter,l2_regularization,random_state,val_mae
0,0.05,31,300,0.0,42,0.098959
1,0.05,63,300,0.0,42,0.096625
2,0.10,31,300,0.0,42,0.100863
3,0.10,63,300,0.0,42,0.097770


deterioration (histgb) chosen params: {'learning_rate': 0.05, 'max_leaf_nodes': 31, 'max_iter': 300, 'l2_regularization': 0.0, 'class_weight': 'balanced', 'random_state': 42}


,learning_rate,max_leaf_nodes,max_iter,l2_regularization,class_weight,random_state,val_auprc
0,0.05,31,300,0.0,balanced,42,0.278038
1,0.05,63,300,0.0,balanced,42,0.269572
2,0.10,31,300,0.0,balanced,42,0.275442
3,0.10,63,300,0.0,balanced,42,0.250785


### 10.3&nbsp;&nbsp;CatBoost (D21) — priority 3, optional dependency

In [19]:
HAVE_CATBOOST = models.catboost_model.CATBOOST_AVAILABLE
if HAVE_CATBOOST:
    catboost_models, catboost_params, catboost_grids = models.catboost_model.train_axis_models_catboost(ft, tg, sp)
    for axis in models.AXES:
        print(f"{axis:7} chosen params: {catboost_params[axis]}")
    det_model_catboost, det_params_catboost, det_grid_catboost = \
        models.catboost_model.train_deterioration_model_catboost(ft, tg, sp)
    print("deterioration (catboost) chosen params:", det_params_catboost)
else:
    print("catboost not installed -- skipping. `pip install catboost` to include this family.")
    catboost_models = None


resp    chosen params: {'learning_rate': 0.05, 'depth': 6, 'iterations': 300, 'random_seed': 42, 'verbose': False}
coag    chosen params: {'learning_rate': 0.05, 'depth': 6, 'iterations': 300, 'random_seed': 42, 'verbose': False}
liver   chosen params: {'learning_rate': 0.05, 'depth': 4, 'iterations': 300, 'random_seed': 42, 'verbose': False}
cardio  chosen params: {'learning_rate': 0.05, 'depth': 6, 'iterations': 300, 'random_seed': 42, 'verbose': False}
neuro   chosen params: {'learning_rate': 0.05, 'depth': 6, 'iterations': 300, 'random_seed': 42, 'verbose': False}
renal   chosen params: {'learning_rate': 0.05, 'depth': 6, 'iterations': 300, 'random_seed': 42, 'verbose': False}
deterioration (catboost) chosen params: {'learning_rate': 0.05, 'depth': 4, 'iterations': 300, 'random_seed': 42, 'verbose': False, 'auto_class_weights': 'Balanced'}


### 10.4&nbsp;&nbsp;LightGBM (D22) — priority 4, optional dependency

In [20]:
HAVE_LIGHTGBM = models.lightgbm_model.LIGHTGBM_AVAILABLE
if HAVE_LIGHTGBM:
    lightgbm_models, lightgbm_params, lightgbm_grids = models.lightgbm_model.train_axis_models_lightgbm(ft, tg, sp)
    for axis in models.AXES:
        print(f"{axis:7} chosen params: {lightgbm_params[axis]}")
    det_model_lightgbm, det_params_lightgbm, det_grid_lightgbm = \
        models.lightgbm_model.train_deterioration_model_lightgbm(ft, tg, sp)
    print("deterioration (lightgbm) chosen params:", det_params_lightgbm)
else:
    print("lightgbm not installed -- skipping. `pip install lightgbm` to include this family.")
    lightgbm_models = None


resp    chosen params: {'learning_rate': 0.05, 'num_leaves': 31, 'n_estimators': 300, 'random_state': 42, 'verbosity': -1}
coag    chosen params: {'learning_rate': 0.05, 'num_leaves': 31, 'n_estimators': 300, 'random_state': 42, 'verbosity': -1}
liver   chosen params: {'learning_rate': 0.05, 'num_leaves': 31, 'n_estimators': 300, 'random_state': 42, 'verbosity': -1}
cardio  chosen params: {'learning_rate': 0.05, 'num_leaves': 31, 'n_estimators': 300, 'random_state': 42, 'verbosity': -1}
neuro   chosen params: {'learning_rate': 0.05, 'num_leaves': 31, 'n_estimators': 300, 'random_state': 42, 'verbosity': -1}
renal   chosen params: {'learning_rate': 0.05, 'num_leaves': 31, 'n_estimators': 300, 'random_state': 42, 'verbosity': -1}
deterioration (lightgbm) chosen params: {'learning_rate': 0.05, 'num_leaves': 63, 'n_estimators': 300, 'random_state': 42, 'verbosity': -1, 'class_weight': 'balanced'}


## 11&nbsp;&nbsp;Results&nbsp;&nbsp;*(TRIPOD+AI 20a–20c, 21, 22, 23a, 23b)*

**20a–20c.** Participant flow: all 12,000 records enter Section 1 and none are excluded
(the dataset's own ≥48 h rule is the only eligibility filter, applied before release); per-
split characteristics were shown in Section 6. **21.** N and outcome-event counts per split ×
horizon were shown in Section 8 (and per-axis N differs slightly between the ordinal and delta
families, since ordinal doesn't require `SOFA_now` to be non-null — reported alongside each
result rather than assumed identical). **22.** The full model specification (chosen
hyperparameters per family) is printed in Section 10 so any of them can be re-fit exactly.
**23a–23b.** Every installed family's performance with bootstrap CIs, **compared against every
other family and the Section 8 baselines**, on **validation** (used for iteration) and on
**test** (unlocked once, after model selection is final).


In [21]:
from functools import reduce

pred_frames = [models.ordinal.predict_axis_scores_ordinal(ordinal_models, ft)]
pred_frames.append(models.linear.predict_axis_scores_linear(linear_models, ft))
pred_frames.append(models.predict_axis_scores(axis_models, ft))
if HAVE_CATBOOST:
    pred_frames.append(models.catboost_model.predict_axis_scores_catboost(catboost_models, ft))
if HAVE_LIGHTGBM:
    pred_frames.append(models.lightgbm_model.predict_axis_scores_lightgbm(lightgbm_models, ft))

all_preds = reduce(lambda a, b: a.merge(b, on=["RecordID", "origin_h"]), pred_frames)
all_preds = all_preds.merge(ft, on=["RecordID", "origin_h"])  # D24: bring sofa_now_<axis> etc. back in
all_preds = all_preds.merge(sp[["RecordID", "split"]], on="RecordID")
full = all_preds.merge(tg, on=["RecordID", "origin_h"])
# D24: per-axis persistence must use the SAME 0-fallback every model column now does
# (see histgb.predict_axis_scores docstring), or it would be scored on a different n.
for axis in models.AXES:
    full[f"persistence_{axis}"] = full[f"sofa_now_{axis}"].fillna(0.0)

TOTAL_COLS = {"persistence": "sofa_now", "ordinal": "sofa_total_pred_ordinal",
             "linear": "sofa_total_pred_linear", "histgb": "sofa_total_pred"}
if HAVE_CATBOOST: TOTAL_COLS["catboost"] = "sofa_total_pred_catboost"
if HAVE_LIGHTGBM: TOTAL_COLS["lightgbm"] = "sofa_total_pred_lightgbm"

for split_name in ("val", "test"):
    d = full[full.split == split_name].dropna(subset=["sofa_total"])
    print(f"--- {split_name} : TOTAL SOFA, all families vs persistence ---")
    display(evaluate.regression_report(d, "sofa_total", TOTAL_COLS))
    for axis in models.AXES:
        axis_cols = {"persistence": f"persistence_{axis}", "ordinal": f"{axis}_pred_ordinal",
                    "linear": f"{axis}_pred_linear", "histgb": f"{axis}_pred"}
        if HAVE_CATBOOST: axis_cols["catboost"] = f"{axis}_pred_catboost"
        if HAVE_LIGHTGBM: axis_cols["lightgbm"] = f"{axis}_pred_lightgbm"
        dd = d.dropna(subset=[f"{axis}_score"])
        print(f"  {axis}:")
        display(evaluate.regression_report(dd, f"{axis}_score", axis_cols, group_col="origin_h"))


--- val : TOTAL SOFA, all families vs persistence ---


,origin_h,series,n,mae,mae_lo,mae_hi,rmse,rmse_lo,rmse_hi
0,24,persistence,1795,0.896936,0.831727,0.959331,1.633562,1.513537,1.739305
1,24,ordinal,1795,0.596157,0.561517,0.629015,0.964262,0.887765,1.035940
2,24,linear,1795,0.762949,0.721266,0.802350,1.161903,1.079185,1.243985
3,24,histgb,1795,0.614113,0.579963,0.648596,0.977096,0.901837,1.049219
4,24,catboost,1795,0.635545,0.599447,0.670542,0.993316,0.917406,1.068147
5,24,lightgbm,1795,0.618722,0.585343,0.651647,0.965468,0.891803,1.033425
6,30,persistence,1794,0.677815,0.624289,0.729668,1.276667,1.183746,1.365485
7,30,ordinal,1794,0.568176,0.534376,0.602618,0.917730,0.850030,1.000013
8,30,linear,1794,0.709011,0.673140,0.745852,1.045765,0.978921,1.118936
9,30,histgb,1794,0.587792,0.553948,0.621104,0.927027,0.860143,1.004746


  resp:


,origin_h,series,n,mae,mae_lo,mae_hi,rmse,rmse_lo,rmse_hi
0,24,persistence,978,0.356851,0.314928,0.402889,0.782607,0.715712,0.853239
1,24,ordinal,978,0.348682,0.317791,0.383656,0.643216,0.577022,0.710166
2,24,linear,978,0.425663,0.395396,0.461581,0.682377,0.620117,0.748398
3,24,histgb,978,0.362729,0.331261,0.398223,0.658670,0.593533,0.724770
4,24,catboost,978,0.377762,0.347417,0.412611,0.652250,0.588578,0.717074
5,24,lightgbm,978,0.368060,0.337111,0.404050,0.654322,0.589097,0.721339
6,30,persistence,895,0.240223,0.204441,0.276006,0.606298,0.536906,0.668526
7,30,ordinal,895,0.288030,0.258948,0.319968,0.538674,0.477505,0.600800
8,30,linear,895,0.366515,0.336104,0.396163,0.583362,0.523933,0.640068
9,30,histgb,895,0.293217,0.263667,0.323735,0.543962,0.480924,0.603163


  coag:


...

,origin_h,series,n,mae,mae_lo,mae_hi,rmse,rmse_lo,rmse_hi
0,24,persistence,1774,0.081736,0.069884,0.094701,0.285895,0.264357,0.307736
1,24,ordinal,1774,0.112564,0.102797,0.122018,0.236677,0.219920,0.253760
2,24,linear,1774,0.150770,0.140605,0.160384,0.255966,0.239532,0.272225
3,24,histgb,1774,0.113297,0.103872,0.122676,0.234940,0.217724,0.251936
4,24,catboost,1774,0.121545,0.111763,0.130936,0.238475,0.220998,0.254706
5,24,lightgbm,1774,0.116027,0.106370,0.125384,0.237107,0.219827,0.254003
6,30,persistence,1773,0.072758,0.061464,0.085730,0.269737,0.247919,0.292798
7,30,ordinal,1773,0.116158,0.106669,0.125811,0.234958,0.218670,0.252387
8,30,linear,1773,0.144781,0.135706,0.154641,0.248393,0.232167,0.266386
9,30,histgb,1773,0.115507,0.106581,0.125557,0.234681,0.218428,0.252307


  neuro:


,origin_h,series,n,mae,mae_lo,mae_hi,rmse,rmse_lo,rmse_hi
0,24,persistence,1773,0.442188,0.399887,0.482798,1.031921,0.960823,1.098376
1,24,ordinal,1773,0.248734,0.229745,0.267971,0.481937,0.432213,0.530716
2,24,linear,1773,0.411922,0.388686,0.435374,0.652816,0.604537,0.695287
3,24,histgb,1773,0.271896,0.252908,0.291194,0.498605,0.452499,0.545349
4,24,catboost,1773,0.301501,0.282668,0.321288,0.523258,0.480725,0.566400
5,24,lightgbm,1773,0.272741,0.254333,0.292105,0.491391,0.445511,0.539035
6,30,persistence,1770,0.293785,0.264407,0.327133,0.748633,0.693757,0.807827
7,30,ordinal,1770,0.237336,0.219108,0.256992,0.455854,0.411387,0.500094
8,30,linear,1770,0.369183,0.349619,0.390634,0.561047,0.524875,0.598040
9,30,histgb,1770,0.256831,0.236857,0.276967,0.477153,0.432162,0.520845


  renal:


,origin_h,series,n,mae,mae_lo,mae_hi,rmse,rmse_lo,rmse_hi
0,24,persistence,1783,0.112170,0.092541,0.134044,0.455539,0.397676,0.509583
1,24,ordinal,1783,0.115421,0.099234,0.131691,0.383936,0.332372,0.434411
2,24,linear,1783,0.159860,0.141842,0.178527,0.435747,0.382096,0.486315
3,24,histgb,1783,0.118915,0.103090,0.135706,0.385883,0.333939,0.435762
4,24,catboost,1783,0.129410,0.113655,0.145312,0.380910,0.331583,0.427399
5,24,lightgbm,1783,0.127272,0.111650,0.142986,0.371640,0.323337,0.417714
6,30,persistence,1784,0.089126,0.073416,0.107623,0.373596,0.321131,0.428131
7,30,ordinal,1784,0.100887,0.087556,0.115152,0.308641,0.264057,0.354673
8,30,linear,1784,0.147366,0.132916,0.163663,0.353021,0.309141,0.397013
9,30,histgb,1784,0.108021,0.094725,0.122480,0.316345,0.272116,0.363361


In [22]:
det_frames = [models.predict_deterioration(det_model, ft)]
det_frames.append(models.linear.predict_deterioration_linear(det_model_linear, ft))
if HAVE_CATBOOST:
    det_frames.append(models.catboost_model.predict_deterioration_catboost(det_model_catboost, ft))
if HAVE_LIGHTGBM:
    det_frames.append(models.lightgbm_model.predict_deterioration_lightgbm(det_model_lightgbm, ft))

det_all = reduce(lambda a, b: a.merge(b, on=["RecordID", "origin_h"]), det_frames)
det_all = det_all.merge(sp[["RecordID", "split"]], on="RecordID")
det_full = det_all.merge(tg[["RecordID", "origin_h", "deteriorate_24h"]], on=["RecordID", "origin_h"]).dropna()

PROB_COLS = {"histgb": "deteriorate_proba", "linear": "deteriorate_proba_linear"}
if HAVE_CATBOOST: PROB_COLS["catboost"] = "deteriorate_proba_catboost"
if HAVE_LIGHTGBM: PROB_COLS["lightgbm"] = "deteriorate_proba_lightgbm"

for split_name in ("val", "test"):
    d = det_full[det_full.split == split_name]
    print(f"--- {split_name} : deterioration classifiers vs each other ---")
    display(evaluate.classification_comparison(d, "deteriorate_24h", PROB_COLS))
    print("  no-skill reference:", evaluate.no_skill_reference(d.deteriorate_24h))
    print("  histgb calibration:")
    display(evaluate.calibration_table(d.deteriorate_24h, d.deteriorate_proba))

print("subgroup MAE (val, total SOFA, histgb) by hemodynamic-instability status:")
display(evaluate.subgroup_report(full[full.split=="val"].dropna(subset=["sofa_total"]),
                                 "sofa_total", "sofa_total_pred", "cardio_instability"))


--- val : deterioration classifiers vs each other ---


,series,n,auroc,auroc_lo,auroc_hi,auprc,auprc_lo,auprc_hi,brier
0,histgb,7179,0.824447,0.796142,0.849468,0.278038,0.224735,0.335257,0.109635
1,linear,7179,0.784445,0.753811,0.809664,0.193183,0.151524,0.241389,0.171260
2,catboost,7179,0.821358,0.795650,0.846492,0.255893,0.205290,0.307457,0.144332
3,lightgbm,7179,0.811448,0.782991,0.836362,0.267280,0.214158,0.324876,0.035975


  no-skill reference: {'prevalence': 0.03858476110878953, 'auroc': 0.5, 'auprc': 0.03858476110878953, 'brier': 0.037095977318967174}
  histgb calibration:


,bin,n,mean_pred,observed_rate
0,"(0.022, 0.0832]",718,0.064664,0.004178
1,"(0.0832, 0.112]",718,0.097826,0.001393
2,"(0.112, 0.139]",718,0.125014,0.009749
3,"(0.139, 0.175]",718,0.156781,0.009749
4,"(0.175, 0.222]",718,0.198314,0.005571
5,"(0.222, 0.278]",717,0.248554,0.032078
6,"(0.278, 0.349]",718,0.312145,0.036212
7,"(0.349, 0.443]",718,0.394331,0.029248
8,"(0.443, 0.568]",718,0.502879,0.073816
9,"(0.568, 0.949]",718,0.675764,0.183844


--- test : deterioration classifiers vs each other ---


,series,n,auroc,auroc_lo,auroc_hi,auprc,auprc_lo,auprc_hi,brier
0,histgb,7187,0.837986,0.815777,0.858395,0.308115,0.260311,0.360842,0.117854
1,linear,7187,0.780247,0.752459,0.806760,0.175367,0.144504,0.213023,0.180387
2,catboost,7187,0.837220,0.816062,0.856427,0.280171,0.231788,0.332425,0.153505
3,lightgbm,7187,0.820461,0.795685,0.842518,0.296700,0.248006,0.345549,0.040823


  no-skill reference: {'prevalence': 0.04508139696674551, 'auroc': 0.5, 'auprc': 0.04508139696674551, 'brier': 0.04304906461427222}
  histgb calibration:


,bin,n,mean_pred,observed_rate
0,"(0.0231, 0.0864]",719,0.067868,0.004172
1,"(0.0864, 0.117]",719,0.102271,0.005563
2,"(0.117, 0.15]",718,0.132437,0.000000
3,"(0.15, 0.189]",719,0.168946,0.018081
4,"(0.189, 0.237]",719,0.212012,0.008345
5,"(0.237, 0.299]",718,0.266807,0.023677
6,"(0.299, 0.375]",719,0.337294,0.031989
7,"(0.375, 0.466]",718,0.418428,0.054318
8,"(0.466, 0.592]",719,0.528149,0.084840
9,"(0.592, 0.943]",719,0.695504,0.219750


subgroup MAE (val, total SOFA, histgb) by hemodynamic-instability status:


,cardio_instability,n,mae,mae_lo,mae_hi
0,0.0,2590,0.592447,0.563522,0.620899
1,1.0,4483,0.591504,0.569141,0.614293


## 12&nbsp;&nbsp;Interpretation — permutation importance&nbsp;&nbsp;(D16)

Run on the **primary family (`histgb`)** only — which family is worth interpreting in depth
is itself a call best made after seeing Section 11's comparison, not before; `evaluate.
permutation_report()` works identically for any fitted model with a `predict`/`predict_proba`
method, so re-running it against whichever family wins is a one-line change once Section 11
has run. `HistGradientBoosting` has no built-in `feature_importances_`; permutation importance
is sklearn's documented alternative. Computed with `n_repeats=3` on a fixed random 2,000-row
subsample of validation (not the full ~33,600-row pooled set) for tractability — a scoping
choice (D16), not a silent shortcut.


In [23]:
cols = models.feature_columns(ft)
for axis, model in axis_models.items():
    d = full[full.split=="val"].dropna(subset=[f"{axis}_delta"])   # D24: delta, not score -- matches what the model was trained/predicts on
    imp = evaluate.permutation_report(model, d[cols], d[f"{axis}_delta"])
    print(f"top features for '{axis}':")
    display(imp)

d = det_full[det_full.split=="val"]
Xd = ft.merge(sp[["RecordID","split"]], on="RecordID")
Xd = Xd[Xd.split=="val"].set_index(["RecordID","origin_h"]).loc[
    d.set_index(["RecordID","origin_h"]).index].reset_index()
imp_det = evaluate.permutation_report(det_model, Xd[cols], d.deteriorate_24h.to_numpy(),
                                      scoring="average_precision")
print("top features for the deterioration classifier:")
display(imp_det)


top features for 'resp':


,feature,importance_mean,importance_std
0,sofa_now_resp,0.881521,0.005406
1,v_PaO2_last,0.201103,0.012864
2,v_FiO2_last,0.109238,0.005723
3,v_MechVent_hrs_since_last,0.076277,0.011449
4,sofa0_24_resp,0.051371,0.004875
5,v_FiO2_slope12h,0.035680,0.003152
6,v_PaO2_min,0.015191,0.005519
7,v_PaO2_slope12h,0.010601,0.004345
8,v_PaO2_first,0.009927,0.001154
9,v_GCS_last,0.009465,0.000943


top features for 'coag':


,feature,importance_mean,importance_std
0,v_Platelets_last,1.763332,0.017184
1,sofa_now_coag,1.116959,0.012452
2,v_Platelets_hrs_since_last,0.171506,0.008297
3,v_Platelets_min,0.034998,0.001720
4,v_Temp_count,0.025352,0.002383
5,v_Platelets_slope12h,0.022726,0.002428
6,v_Platelets_mean,0.016554,0.005439
7,sofa0_24_coag,0.007466,0.002392
8,v_SysABP_last,0.005482,0.000341
9,v_SysABP_mean,0.004572,0.000880


top features for 'liver':


...

,feature,importance_mean,importance_std
0,sofa_now_neuro,0.923621,0.039348
1,v_GCS_last,0.197574,0.008555
2,v_GCS_mean,0.169039,0.003667
3,v_GCS_std,0.134275,0.021710
4,v_GCS_first,0.058309,0.005817
5,v_GCS_slope12h,0.051903,0.001654
6,v_FiO2_hrs_since_last,0.044638,0.006815
7,v_MechVent_hrs_since_last,0.033052,0.002677
8,v_PaCO2_hrs_since_last,0.027669,0.007503
9,v_GCS_hrs_since_last,0.018704,0.004476


top features for 'renal':


,feature,importance_mean,importance_std
0,sofa_now_renal,4.171223,0.110439
1,v_Creatinine_last,2.305187,0.052183
2,e_urine_sum_24h,0.084942,0.011978
3,v_Creatinine_hrs_since_last,0.077468,0.002577
4,v_Urine_last,0.050351,0.001582
5,v_Urine_slope12h,0.009060,0.003410
6,v_Mg_hrs_coverage,0.004677,0.002188
7,v_K_hrs_since_last,0.004497,0.002051
8,v_Creatinine_max,0.004474,0.001384
9,v_PaCO2_first,0.004336,0.001513


top features for the deterioration classifier:


,feature,importance_mean,importance_std
0,sofa_now_resp,0.103683,0.008998
1,sofa_now_renal,0.065401,0.003157
2,v_FiO2_hrs_since_last,0.045913,0.020959
3,v_HR_hrs_since_last,0.035396,0.007094
4,e_urine_sum_24h,0.020403,0.008017
5,v_GCS_hrs_since_last,0.018771,0.004220
6,v_PaCO2_hrs_since_last,0.017901,0.004026
7,v_PaO2_min,0.016248,0.003581
8,v_BUN_hrs_since_last,0.008667,0.001016
9,v_GCS_std,0.008305,0.005326


## 13&nbsp;&nbsp;Discussion&nbsp;&nbsp;*(TRIPOD+AI 25, 26, 27a–27c)*

**25 — overall interpretation.** Four of the five families — ordinal (D19), histgb
(primary), catboost (D21) and lightgbm (D22) — beat the persistence baseline on total SOFA at
**every one of the 4 horizons, on both validation and test**, comfortably clearing the
project's success criterion (docx §1.6: "beats persistence at >= 2 of 4 horizons"). The
fifth, the **plain linear/Ridge baseline (D20), does not**: it beats persistence only at h24
and is *worse* than persistence at h30/h36/h42 on both splits — not a bug, but the direct
answer to the question D20 was built to ask ("does the tree ensembles' non-linearity earn its
keep"): yes, clearly — a plain regularized linear model reverts too far toward the mean delta
at the longer horizons, where persistence itself is already a strong, hard-to-beat guess. The
headline surprise: **the ordinal decomposition (D19) is the best-performing family at every
horizon**, not merely a methodologically-cleaner alternative — e.g. total-SOFA test MAE at
h24: persistence 0.96, linear 0.80, histgb 0.65, catboost 0.68, **ordinal 0.63**. Among the
three GBDT engines, **HistGB (the original primary family, D12) remains the best or
statistically tied for best** at every horizon — CatBoost's ordered boosting and LightGBM's
leaf-wise growth do not clearly improve on it here, so the added dependency is not obviously
worth carrying forward. Per axis, the picture is uneven: models add real, substantial value
for **neuro, renal and coagulation**; for **respiratory, liver and cardiovascular**,
persistence is competitive with or sometimes better than every trained family (plausible
reasons: cardio is binary and pressor-driven MAP swings look like noise to a model with no
drug data; resp/liver have the sparsest, most irregularly-timed measurements of the six axes,
per Section 5's missing-data profile). Fairness (item 25): as expected, the cardiovascular
axis itself stays capped at 0/1 for every family (no vasopressor data) — see Section 9's
fairness note and `SOFA_modelling_approaches.docx`.

The `deteriorate_24h` classifier shows real, useful discrimination for every family with a
classifier variant (histgb, linear, catboost, lightgbm — ordinal has no deterioration
classifier of its own; AUROC 0.78–0.84, AUPRC 5–8x the no-skill reference of ~0.04) but is
**not well calibrated in absolute terms** — the top predicted-probability decile averages
~0.68 while the observed event rate there is only ~0.18–0.22 (Section 11's calibration
table): the ranking is useful, the raw probability is not yet a trustworthy percentage.
LightGBM's Brier score is essentially tied with the no-skill reference despite good
AUROC/AUPRC — a sign its probabilities are systematically under-scaled even though its
ranking is competitive; this would need explicit recalibration (e.g. Platt/isotonic) before
using LightGBM's raw probability output for anything beyond ranking.

**Two bugs were found only because this was actually executed, not just read (D23, D24) —
worth stating plainly, since it's the entire reason "run it and debug it" was asked for
rather than trusting the code review alone:**
- **D23**: CatBoost rejects `cat_features` columns typed as float64/NaN, even for
  clean small-integer codes like `origin_h = 24.0`; fixed by casting to string with an
  explicit missing-value sentinel, applied identically at fit and predict time.
- **D24**: the four delta-based families' predictions silently went `NaN` whenever
  `sofa_now_<axis>` was missing (0.9–60% of rows depending on axis), which propagated into
  `sofa_total_pred` and restricted every downstream comparison to the ~22% of rows with all
  six axes simultaneously measured — a much easier population than the full one persistence
  and the ordinal family were being scored on. Before the fix, this made the four delta
  families look *better* than they should have on an easy subset, and made ordinal (which
  never produces `NaN`) look catastrophically worse than persistence on the full, harder
  population purely from an apples-to-oranges comparison. Fixed by gating every family's
  per-axis prediction to fall back to 0 when `sofa_now_<axis>` is missing — mirroring the
  same lenient "missing axis -> 0" convention `sofa_total` itself already uses (D10) — applied
  uniformly to all five families so the comparison is on the same population throughout.

**26 — limitations.**
- The cardiovascular axis cannot exceed 1 anywhere in this project (no vasopressor data);
  every downstream number inherits that ceiling on both the feature and target side.
- 18 of the 24 target-window hours are already observed at prediction time, so all reported
  MAEs include a large "nothing changed" component; only the delta figures isolate genuine
  forecast skill.
- No accrual dates exist in the de-identified data, so temporal drift cannot be assessed.
- Single-centre-era, single-dataset internal validation only — no external validation.
- `deteriorate_24h` is rare (~4 %); classifier performance estimates are comparatively
  imprecise (see its CI width), and none of the four classifiers is well calibrated in
  absolute-probability terms (above).
- The ordinal family's win is measured under a gate (D24) that forces it to also predict 0
  when `sofa_now_<axis>` is missing, even though its classifiers could, in principle, guess
  from other features without that anchor — its real advantage over the other families may
  be under- rather than over-stated here.
- The linear baseline (D20) losing to persistence at h30/h36/h42 is a property of a *plain*
  Ridge model on this delta target — it does not mean linear methods in general cannot work
  here, only that this specific, deliberately-simple configuration cannot.

**27a — handling poor-quality/unavailable input at use time.** If a required vital has no
recent measurement, `HistGradientBoosting`/`LightGBM` route it through their native
missing-value branch rather than substituting a value, and every family's total-score
reconstruction falls back to the documented 0-default (D24) when `sofa_now_<axis>` itself is
absent — a user of this model should still be told which inputs were absent for a given
prediction (the `*_present` / `*_count` features already carry that signal and should be
surfaced, not just consumed silently).
**27b — user interaction required.** None beyond ensuring the input record covers `[0,T)`
with the same variable definitions used here; this is a decision-support estimate, not an
autonomous action.
**27c — next steps.** External validation on a different ICU/era; incorporating real
vasopressor data if it ever becomes available for this cohort; a fairness audit stratified by
age and sex once a larger cohort permits stable subgroup estimates; recalibrating the
deterioration classifiers (Platt/isotonic) before reporting the raw probability as a risk
percentage; an ablation re-running the ordinal family without D24's gate, to quantify what
its "predict from indirect evidence" capability is worth on its own; extending the ordinal
decomposition with a monotonicity constraint across thresholds (not enforced in this
version); given HistGB already matches or beats CatBoost/LightGBM, dropping the extra
dependencies unless a future data expansion changes that; a lightly-regularized or
feature-interaction-aware linear variant, to check whether D20's loss to persistence is a
property of linear models here or just of the plain Ridge configuration tried so far.

**Open Science — item 18e–18f.** Data: PhysioNet/CinC Challenge 2012 (public,
physionet.org). Code: this repository (`github.com/398-21/healthcare-ai-assin`), including
`modeling/data_prep/`, `modeling/models/` (`common.py`, `ordinal.py`, `linear.py`,
`histgb.py`, `catboost_model.py`, `lightgbm_model.py`), `modeling/evaluate.py` and this
notebook.

## 14&nbsp;&nbsp;Save artefacts&nbsp;&nbsp;*(TRIPOD+AI 22)*

Saves every family that was actually trained in Section 10 (all 5 if CatBoost/LightGBM are
installed on the training device, 3 otherwise), under `outputs/models/<family>/`, plus one
`model_spec.json` recording every family's chosen hyperparameters, the feature list and the
config used -- so any of them can be re-fit exactly (item 22).


In [24]:
import json as _json

out_models = config.OUT_DIR / "models"
spec = {"seed": config.SEED, "horizons": list(config.HORIZONS_H),
       "feature_columns": models.feature_columns(ft), "families": {}}

def _save_family(name, axis_models_, axis_params_, det_model_=None, det_params_=None):
    d = out_models / name
    d.mkdir(parents=True, exist_ok=True)
    for axis, m in axis_models_.items():
        joblib.dump(m, d / f"axis_{axis}.joblib")
    fam_spec = {"axis_params": axis_params_}
    if det_model_ is not None:
        joblib.dump(det_model_, d / "deteriorate_24h.joblib")
        fam_spec["deterioration_params"] = det_params_
    spec["families"][name] = fam_spec
    print(f"saved '{name}': {len(axis_models_)} axis models"
         f"{' + deterioration classifier' if det_model_ is not None else ''} -> {d}")

_save_family("ordinal", ordinal_models, ordinal_params)   # no single deterioration model (D19 is axis-only)
_save_family("linear", linear_models, linear_params, det_model_linear, det_params_linear)
_save_family("histgb", axis_models, axis_params, det_model, det_params)
if HAVE_CATBOOST:
    _save_family("catboost", catboost_models, catboost_params, det_model_catboost, det_params_catboost)
if HAVE_LIGHTGBM:
    _save_family("lightgbm", lightgbm_models, lightgbm_params, det_model_lightgbm, det_params_lightgbm)

(config.OUT_DIR / "model_spec.json").write_text(_json.dumps(spec, indent=2))
print("\nwrote", config.OUT_DIR / "model_spec.json")


saved 'ordinal': 6 axis models -> C:\Users\pan shengxin\Downloads\release\release\healthcare-ai-assin\modeling\outputs\models\ordinal
saved 'linear': 6 axis models + deterioration classifier -> C:\Users\pan shengxin\Downloads\release\release\healthcare-ai-assin\modeling\outputs\models\linear
saved 'histgb': 6 axis models + deterioration classifier -> C:\Users\pan shengxin\Downloads\release\release\healthcare-ai-assin\modeling\outputs\models\histgb
saved 'catboost': 6 axis models + deterioration classifier -> C:\Users\pan shengxin\Downloads\release\release\healthcare-ai-assin\modeling\outputs\models\catboost
saved 'lightgbm': 6 axis models + deterioration classifier -> C:\Users\pan shengxin\Downloads\release\release\healthcare-ai-assin\modeling\outputs\models\lightgbm

wrote C:\Users\pan shengxin\Downloads\release\release\healthcare-ai-assin\modeling\outputs\model_spec.json
